# Phulax LoRA fine-tune (Colab T4)

Mirrors `ml/finetune/lora.py` byte-for-byte on hyperparams, prompt template, loss masking, and class weighting. Runs on a Colab T4 in ~5-15 min instead of 3-6h on CPU.

**Before running:** in Colab, set Runtime → Change runtime type → T4 GPU.

**Inputs:** `dataset.jsonl` (uploaded in cell 2).
**Output:** `adapter_model.safetensors` + `training_meta.json` zipped and downloaded to your laptop. Drop both into `ml/artifacts/lora/` locally; `ml/finetune/merge_and_quantize.py` picks up from there.

In [ ]:
# 1. Install pinned deps. ~90s on a fresh runtime.
!pip install -q "transformers==4.46.3" "peft==0.13.2" "datasets==3.1.0" "accelerate==1.1.1" "safetensors>=0.4.5" "bitsandbytes>=0.44.1"
import torch
assert torch.cuda.is_available(), "No GPU detected. Runtime → Change runtime type → T4 GPU."
print(torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# 2. Upload dataset.jsonl from your laptop.
# Use Colab's file picker; pick ml/data/dataset.jsonl.
from google.colab import files
uploaded = files.upload()
assert "dataset.jsonl" in uploaded, "upload dataset.jsonl exactly"
import pathlib
DATA = pathlib.Path("dataset.jsonl")
OUT = pathlib.Path("adapter_out"); OUT.mkdir(exist_ok=True)
print("rows:", sum(1 for _ in DATA.open()))

In [ ]:
# 3. Frozen prompt template (verbatim copy of ml/prompt/template.py @ TEMPLATE_VERSION 2.0.0).
# Keep in sync if you bump the version locally.
import json
from typing import Any

TEMPLATE_VERSION = "2.0.0"
SIGNALS = ("none","drain","oracle_manipulation","donation_attack","reentrancy","governance_hijack","sig_bypass","share_inflation")
SYSTEM = (
    "You are Phulax, a transaction risk classifier for DeFi lending pools. "
    "You receive a feature blob describing a single on-chain transaction, "
    "including the caller's role and history (timelock, multisig, EOA, "
    "contract), the function signature, the decoded arguments, and the "
    "resulting balance-delta vector. "
    "Output ONLY a single JSON object of the form "
    '{"p_nefarious": <float in [0,1]>, "signal": <one of '
    '"none","drain","oracle_manipulation","donation_attack","reentrancy",'
    '"governance_hijack","sig_bypass","share_inflation">}. '
    "No prose, no markdown, no explanation. p_nefarious is a calibrated "
    "probability, not a binary tag — use intermediate values when the "
    "transaction is suspicious-shaped but executed by a trusted caller "
    "(timelock, governance multisig with high quorum). signal == \"none\" "
    "means SAFE."
)

def canonicalise(row):
    fb = {
        "caller": row.get("caller", {"role":"unknown","age_days":0,"signer_quorum":None}),
        "selector": row["selector"],
        "fn": row["fn"],
        "decoded_args": row["decoded_args"],
        "balance_delta": row["balance_delta"],
    }
    return json.dumps(fb, sort_keys=True, separators=(",", ":"))

def user_message(row): return f"Classify this transaction:\n{canonicalise(row)}"

def signal_for(row):
    sig = row.get("signal")
    if sig in SIGNALS: return sig
    return "none" if row.get("label", "SAFE") == "SAFE" else "drain"

def assistant_target(row):
    if "risk_score" in row:
        p = float(row["risk_score"])
    else:
        p = 0.94 if row.get("label") == "RISK" else 0.03
    p = round(max(0.0, min(1.0, p)), 3)
    return json.dumps({"p_nefarious": p, "signal": signal_for(row)}, separators=(",", ":"))

def chat_messages(row, with_target=True):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_message(row)},
    ]
    if with_target:
        msgs.append({"role": "assistant", "content": assistant_target(row)})
    return msgs

In [ ]:
# 4. Hyperparams (locked to lora.py).
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]
LR = 2e-4
EPOCHS = 10
SEED = 1337
MAX_LEN = 768
RISK_WEIGHT = 2.0
SAFE_WEIGHT = 1.0

In [ ]:
# 5. Load + split dataset (deterministic, matches lora.py).
import json, random
rows = [json.loads(line) for line in DATA.read_text().splitlines() if line.strip()]
rng = random.Random(SEED)
rng.shuffle(rows)
cut = int(len(rows) * 0.8)
train, eval_ = rows[:cut], rows[cut:]
print(f"train={len(train)} eval={len(eval_)}")

In [ ]:
# 6. Tokenise with assistant-only loss masking + per-row class weights.
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def encode(row):
    prefix_text = tok.apply_chat_template(chat_messages(row, with_target=False), tokenize=False, add_generation_prompt=True)
    full_text = tok.apply_chat_template(chat_messages(row, with_target=True), tokenize=False, add_generation_prompt=False)
    prefix_ids = tok(prefix_text, add_special_tokens=False)["input_ids"]
    full = tok(full_text, truncation=True, max_length=MAX_LEN, padding="max_length", add_special_tokens=False)
    labels = list(full["input_ids"])
    prefix_len = min(len(prefix_ids), len(labels))
    for i in range(prefix_len):
        labels[i] = -100
    pad_id = tok.pad_token_id
    for i, tid in enumerate(labels):
        if tid == pad_id:
            labels[i] = -100
    full["labels"] = labels
    full["class_weight"] = RISK_WEIGHT if row.get("label") == "RISK" else SAFE_WEIGHT
    return full

train_ds = Dataset.from_list(train).map(encode, remove_columns=list(train[0]))
eval_ds = Dataset.from_list(eval_).map(encode, remove_columns=list(eval_[0]))
sample = train_ds[0]
n_sup = sum(1 for x in sample["labels"] if x != -100)
print(f"loss-mask check: {n_sup}/{len(sample['labels'])} tokens supervised (~20-30 expected)")

In [ ]:
# 7. Build model + LoRA. Uses fp16 on T4 (no bf16 support); will use bf16 on A100/L4 if you upgraded.
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype, device_map="auto")
lora = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                  target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 8. WeightedTrainer + WeightedCollator (verbatim from lora.py).
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        class_weight = inputs.pop("class_weight", None)
        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs["labels"]
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fn = torch.nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
        per_token = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)).view(shift_labels.size())
        mask = (shift_labels != -100).float()
        if class_weight is not None:
            w = class_weight.to(per_token.dtype).view(-1, 1)
            per_token = per_token * w
        loss = (per_token * mask).sum() / mask.sum().clamp(min=1)
        return (loss, outputs) if return_outputs else loss

class WeightedCollator:
    def __init__(self, base): self._base = base
    def __call__(self, features):
        weights = [float(f.pop("class_weight", 1.0)) for f in features]
        batch = self._base(features)
        batch["class_weight"] = torch.tensor(weights, dtype=torch.float32)
        return batch

args = TrainingArguments(
    output_dir=str(OUT),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=LR,
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=SEED,
    report_to=[],
)
trainer = WeightedTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
    processing_class=tok,
    data_collator=WeightedCollator(DataCollatorForLanguageModeling(tok, mlm=False)),
)

In [ ]:
# 9. Train. T4 wallclock for ~325 train rows, 10 epochs, MAX_LEN 768: ~6-12 min.
trainer.train()
trainer.save_model(str(OUT))
tok.save_pretrained(str(OUT))
import json
(OUT / "training_meta.json").write_text(json.dumps({
    "base_model": BASE_MODEL, "rank": LORA_RANK, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES, "lr": LR, "epochs": EPOCHS, "seed": SEED,
    "train_size": len(train), "eval_size": len(eval_),
    "template_version": TEMPLATE_VERSION,
}, indent=2))
print("adapter contents:", sorted(p.name for p in OUT.iterdir()))

In [ ]:
# 10. Zip + download. Drop the unzipped contents into ml/artifacts/lora/ on your laptop.
import shutil
zip_path = shutil.make_archive("phulax_lora_adapter", "zip", root_dir=str(OUT))
from google.colab import files
files.download(zip_path)
print("downloaded:", zip_path)